In [4]:
!pip install openclean

In [96]:
from ipywidgets.widgets.trait_types import date_from_json
from openclean.pipeline import stream
from openclean.profiling.column import DefaultColumnProfiler
from openclean.profiling.anomalies.sklearn import DBSCANOutliers
import pandas as pd
import numpy as np

from ipywidgets import interact, interactive
import ipywidgets as widgets
from IPython.display import display
# messy file example path: "sample_data/messy_ecommerce_sales_data.csv"

title = widgets.HTML(value="<h2 style='font-size: 24px; margin: 5px; color: pink; justify_content: center;'>Data Profiling and Cleaning Using Openclean</h2>")
sub_title = widgets.HTML(value="<h2 style='font-size: 12px; margin: 0px; color: lightblue; justify_content: center;'>Column Profile</h2>")
sub_title_2 = widgets.HTML(value="<h2 style='font-size: 12px; margin: 0px; color: lightblue; justify_content: center;'>Column Profile - Min and Max Values by Data Type</h2>")
sub_title_3 = widgets.HTML(value="<h2 style='font-size: 12px; margin: 0px; color: lightblue; justify_content: center;'>Specify Column to Delete Null Rows</h2>")
sub_title_4 = widgets.HTML(value="<h2 style='font-size: 12px; margin: 0px; color: lightblue; justify_content: center;'>Delete Duplicates</h2>")

ds_state = {"ds_full": stream("sample_data/messy_ecommerce_sales_data.csv")}

preview_button = widgets.Button(description="Preview Data")
output = widgets.Output()
save_output = widgets.Output()

preview_state = {"shown": False}
clean_state = {"shown": False}

save_input = widgets.Text(
    placeholder='New File Name',
    description=''
)

def save_file(b):
  with save_output:
    save_output.clear_output()
    try:
      df = ds_state["ds_full"].to_df()
      df.to_csv(save_input.value + ".csv", index=False)
      print("File saved")
    except Exception as e:
      save_output.clear_output()
      print(e)

save_button = widgets.Button(description="Save File")
save_button.on_click(save_file)

def find_null_entries(column):
  try:
    df = ds_state['ds_full'].to_df()
    df[column] = df[column].replace(['', ' ', 'NA', 'null'], np.nan)
    df = df.dropna(subset=[column])
    ds_state["ds_full"] = stream(df)
    print("Rows with the null value in the " + column + " column were deleted.")
  except Exception as e:
    print(e)

clean_button = widgets.Button(description="Clean Data")
remove_duplicate_button = widgets.Button(description="Remove Duplicates")

def remove_duplicates(b):
  with output:
    try:
      df = ds_state['ds_full'].to_df()
      df = df.drop_duplicates()
      ds_state["ds_full"] = stream(df)
      print("Duplicate rows were were deleted.")
    except Exception as e:
      print(e)

column_null = interactive(find_null_entries, column='enter a column name', layout = widgets.Layout(padding="50px", align="200px"))

column_old_name = widgets.Text(
    placeholder='Enter old column name',
    description='Old Name:'
)

column_new_name = widgets.Text(
    placeholder='Enter new column name',
    description='New Name:'
)

rename_button = widgets.Button(description="Rename Column")
def rename_column(b):
  with output:
    try:
      df = ds_state['ds_full'].to_df()
      df = df.rename(columns={column_old_name.value: column_new_name.value})
      ds_state["ds_full"] = stream(df)
      print("Column name was updated")
    except Exception as e:
      print(e)

rename_button.on_click(rename_column)

rename_screen = widgets.HBox(
    children = [column_old_name, column_new_name, rename_button],
    layout = widgets.Layout(width="auto", padding="15px")
)

save_screen = widgets.HBox(
    children = [save_input, save_button],
    layout = widgets.Layout(width="auto", padding="15px")
)

cleaningScreen = widgets.VBox(
    children = [sub_title_3, column_null, sub_title_4, remove_duplicate_button, rename_screen, save_screen, save_output],
    layout = widgets.Layout(width="auto", padding="15px")
)

def on_button_clicked_clean(b):
   with output:
      if not clean_state["shown"]:
        preview_state["shown"] = False
        try:
          clean_state["shown"] = True
          display(cleaningScreen)
        except Exception as e:
          print(e)
      else:
        output.clear_output()
        save_output.clear_output()
        clean_state["shown"] = False

clean_button.on_click(on_button_clicked_clean)
remove_duplicate_button.on_click(remove_duplicates)

profile_output = widgets.Output()

def find_profile(b):
 with profile_output:
  profile_output.clear_output()
  try:
    column = ds_state["ds_full"].select(columns=profile_input.value)
    column_profile = column.profile(default_profiler=DefaultColumnProfiler)
    profile = column_profile.stats()
    maxmin = column_profile.minmax(profile_input.value)
    display(sub_title, profile, sub_title_2, maxmin)
  except Exception as e:
    print(e)

profile_input = widgets.Text(
    placeholder='Enter column name',
    description=''
)

profile_button = widgets.Button(description="Profile Column")
profile_button.on_click(find_profile)

profile_screen = widgets.HBox(
    children = [profile_input, profile_button],
    layout = widgets.Layout(width="auto", padding="15px")
)

profile_screen = widgets.VBox(
    children = [profile_screen, profile_output],
    layout = widgets.Layout(width="auto", padding="15px")
)

def on_button_clicked(b):
    with output:
      if not preview_state["shown"]:
        clean_state["shown"] = False
        try:
          display(ds_state["ds_full"].head())
          preview_state["shown"] = True
          display(profile_screen)
        except Exception as e:
          preview_state["shown"] = True
          print(e)
      else:
        output.clear_output()
        profile_output.clear_output()
        preview_state["shown"] = False

preview_button.on_click(on_button_clicked)

def find_file(filepath):
  with output:
    try:
      ds_state["ds_full"] = stream(filepath)
      ds_state["ds_full"].head()
      output.clear_output()
      preview_state["shown"] = False
      clean_state["shown"] = False
    except Exception as e:
      output.clear_output()
      print("file not found")


def save_file(b):
  df = ds_state["ds_full"].to_df()
  df.to_csv("cleaned_data.csv", index=False)

file_input = interactive(find_file, filepath='sample_data/messy_ecommerce_sales_data.csv', layout = widgets.Layout(width="900px", padding="50px", align="200px"))

button_screen = widgets.HBox(
    children = (preview_button, clean_button),
    layout = widgets.Layout(width="auto", padding="15px", justify_content="center")
)

home_screen = widgets.VBox(
    children = (title, file_input, button_screen, output),
    layout = widgets.Layout(width="1000px", padding="10px")
)


display(home_screen)
